---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: Web Analytics

### 📋 **Topic**: Text Analytics

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

> **Before you begin:** [Update your course files and Python environment safely.](https://github.com/apostolosfilippas/wa/blob/main/assets/before-you-begin.md)



# 🚪 1. Introduction

As always, we'll import the packages we'll need for this notebook.

In [ ]:
from pathlib import Path
import nltk
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Find the course folder, whether we started in wa, scripts, or homework
course_folder = Path.cwd().resolve()
if course_folder.name in {'scripts', 'homework'}:
    course_folder = course_folder.parent
if not (course_folder / 'pyproject.toml').is_file():
    raise RuntimeError('Open the wa course folder in VS Code, then restart this notebook.')
data_folder = course_folder / 'data'

## Data
The file that I've placed in `data/books.csv` contains 2,000 Amazon book reviews.

The data set contains two columns:

- `review_text`: the review itself.
- `positive`: 1 for a positive review, 0 for a negative one.

Let's read it into a pandas data frame, just as we did in the previous lectures.

You'll notice two arguments in `pd.read_csv()`: `quotechar` tells pandas that double quotes surround text fields, and `escapechar` tells it how quotes inside those fields are escaped. The extra backslash in the Python string is how we write a literal backslash.

In [ ]:
data = pd.read_csv(data_folder / 'books.csv', quotechar='"', escapechar='\\')
data.head(20)

In [ ]:
data.iloc[50]['review_text']

# 📝🏷️ 2. Text classification
We are going to look at some Amazon reviews and classify them into positive or negative.

## From text to numbers
Going from text to numeric data is very easy. Let's take a look at how we can do this. We'll start by separating out our X and Y data.

Before we throw thousands of words into a model, let's look at three tiny reviews.

Each **row** is a review and each **column** is a word. Binary features tell us whether a word appears; counts tell us how many times it appears. TF-IDF gives words weights instead. Look at `good` in the first row!

In [ ]:
tiny_reviews = ['good good book', 'bad book', 'good story']

tiny_tables = {}
for name, tiny_vectorizer in {
    'Binary': CountVectorizer(binary=True),
    'Counts': CountVectorizer(),
    'TF-IDF': TfidfVectorizer(),
}.items():
    tiny_matrix = tiny_vectorizer.fit_transform(tiny_reviews)
    tiny_tables[name] = pd.DataFrame(
        tiny_matrix.toarray(), index=tiny_reviews,
        columns=tiny_vectorizer.get_feature_names_out()
    )

pd.concat(tiny_tables, names=['Representation', 'Review']).round(3)

In [ ]:
X_text = data['review_text']
Y = data['positive']

Next, we will turn `X_text` into just `X` -- a numeric representation!

In [ ]:
# Create a vectorizer that will track text as binary features
binary_vectorizer = CountVectorizer(binary=True)

# Let the vectorizer learn what tokens exist in the text data
binary_vectorizer.fit(X_text)

# Turn these tokens into a numeric matrix
X = binary_vectorizer.transform(X_text)

In [ ]:
X

## Modeling
We have a ton of features, let's use them in some different models.

In [ ]:
# Create a model
logistic_regression = LogisticRegression()

# Use this model and our data to get 5-fold cross validation
accs = cross_val_score(logistic_regression, X, Y, scoring="accuracy", cv=5)

# Print out the average accuracy rounded to three decimal points
print(f"The accuracy of our classifier is {np.mean(accs):.3f}")

Let's try using full counts instead of a binary representation. I've just copy and pasted what is above and removed the `binary=True` from the vectorizer.

In [ ]:
# Create a vectorizer that will track text as counted features
count_vectorizer = CountVectorizer()

# Let the vectorizer learn what tokens exist in the text data
count_vectorizer.fit(X_text)

# Turn these tokens into a numeric matrix
X = count_vectorizer.transform(X_text)

# Create a model
logistic_regression = LogisticRegression(max_iter=100000)

# Use this model and our data to get 5-fold cross validation accuracy
accs = cross_val_score(logistic_regression, X, Y, scoring="accuracy", cv=5)

# Print out the average accuracy rounded to three decimal points
print(f"The accuracy of our classifier is {np.mean(accs):.3f}")

Let's try using TF-IDF (term frequency–inverse document frequency).

The idea: a word that appears often in one review, but rarely across the other reviews, might tell us more about that review. Words that appear everywhere get less weight. These are weights, not just word counts. Scikit-learn also normalizes each review's vector by default.

In [ ]:
# Create a vectorizer that will represent text using TF-IDF weights
tfidf_vectorizer = TfidfVectorizer()

# Let the vectorizer learn what tokens exist in the text data
tfidf_vectorizer.fit(X_text)

# Turn these tokens into a numeric matrix
X = tfidf_vectorizer.transform(X_text)

# Create a model
logistic_regression = LogisticRegression(max_iter=100000)

# Use this model and our data to get 5-fold cross validation accuracy
accs = cross_val_score(logistic_regression, X, Y, scoring="accuracy", cv=5)

# Print out the average accuracy rounded to three decimal points
print(f"The accuracy of our classifier is {np.mean(accs):.3f}")

# 🔍🔧 3. Feature Engineering (optional extension)

Text isn't the only thing we can turn into numeric features. Here's a quick detour using five made-up records. We'll create the data right here, so there is no extra file to download.

We'll look at binarizing categories, giving ordered categories numeric values, and creating a new feature. These aren't all the same operation!

In [ ]:
data_dict = {
    'Minutes': [100, 220, 500, 335, 450],
    'Gender': ['Male', 'Female', 'Female', 'Male', 'Male'],
    'Marital': ['Single', 'Married', 'Divorced', 'Single', 'Married'],
    'Satisfaction': ['Low', 'Very Low', 'High', 'Neutral', 'Very High'],
    'Churn': [0, 0, 1, 0, 1]
}

data = pd.DataFrame(data_dict)

data.head()

## Binarizing
Get a list of features you want to binarize, go through each feature and create new features for each level.

In [ ]:
features_to_binarize = ["Gender", "Marital"]

# Get dummies for the desired columns and drop the first category for each
data_dummies = pd.get_dummies(data[features_to_binarize], drop_first=True)

# Drop the original columns from the data
data = data.drop(features_to_binarize, axis=1)

# Concatenate the original data and the dummies
data = pd.concat([data, data_dummies], axis=1)

data.head()

## Ordered categories
We can replace text levels with a numeric mapping we create. This is **ordinal encoding**, not numeric scaling: we are giving an ordered category numbers, not rescaling an existing numeric feature.

Here, satisfaction has a natural order. Our choice of -2 through 2 also treats the gaps as equal; that is a modeling choice, not something the data tells us.

In [ ]:
satisfaction_values = {'Very Low': -2, 'Low': -1, 'Neutral': 0, 'High': 1, 'Very High': 2}
data['Satisfaction'] = data['Satisfaction'].map(satisfaction_values)
data.head()

## Creating new features

A linear model doesn't have to use just the original features. Adding `Minutes_squared` lets it represent a curved relationship.

Does it help? We don't know yet. We'd have to compare performance on validation data, rather than assume that adding a feature makes the model better.

In [ ]:
data['Minutes_squared'] = data['Minutes'] ** 2
data.head()

Feature engineering is an important part of creating a machine learning model. It's a process that requires a lot of creativity and domain knowledge. It's also a process that can be automated to some extent.

# 4. More text preprocessing

## N-grams

N-grams are contiguous sequences of n items (words, letters) from a given sample of text. Beyond just individual words (1-grams or unigrams), using 2-grams, 3-grams, etc., helps in capturing phrases and the context in which words appear. For instance, "not good" as a 2-gram has a different sentiment than the individual words "not" and "good".

In [ ]:
data = pd.read_csv(data_folder / 'books.csv', quotechar='"', escapechar='\\')
text_data = data['review_text']

# Using just 1-grams (individual words)
unigram_vectorizer = CountVectorizer()
X_unigrams = unigram_vectorizer.fit_transform(text_data)

# Using BOTH 1-grams and 2-grams; (2, 2) would mean only 2-grams
bigram_vectorizer = CountVectorizer(ngram_range=(1, 2))
X_bigrams = bigram_vectorizer.fit_transform(text_data)

# Notice that the number of columns greatly increased
print(X_unigrams.shape)
print(X_bigrams.shape)

## Part-of-Speech (POS) Tagging (optional extension)

POS tagging labels words as nouns, verbs, adjectives, etc. Let's look at what it does to one review.

`uv sync` installed NLTK, the Python package. NLTK also needs some language data. We'll download only the resources used below, not its entire collection. The first run needs an internet connection; later runs reuse the downloaded files in `temp/nltk_data`. Run this setup cell for the stemming and stop-word examples too, even if you skip POS tagging.

In [ ]:
# Keep NLTK's downloaded data in our course folder
nltk_folder = course_folder / 'temp' / 'nltk_data'
nltk_folder.mkdir(parents=True, exist_ok=True)
if str(nltk_folder) not in nltk.data.path:
    nltk.data.path.insert(0, str(nltk_folder))

resources = {
    'punkt_tab': 'tokenizers/punkt_tab/english/',
    'averaged_perceptron_tagger_eng': 'taggers/averaged_perceptron_tagger_eng/',
    'wordnet': 'corpora/wordnet.zip',
    'stopwords': 'corpora/stopwords',
    'cmudict': 'corpora/cmudict',  # also used by the readability example
}

for package, resource in resources.items():
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(package, download_dir=str(nltk_folder), quiet=True, raise_on_error=True)

In [ ]:
sample_text = text_data.iloc[0]
tokens = nltk.word_tokenize(sample_text)
pos_tags = nltk.pos_tag(tokens)

print(sample_text)
# Show just the first 20 tags, so the output doesn't take over the notebook
for token, pos in pos_tags[:20]:
    print(f"{token} - {pos}")

## Text Complexity & Readability (optional extension)

Readability scores give us a rough indication of how easy a passage is to read. Here we use **Flesch Reading Ease**: higher scores mean easier reading. This is not the Flesch–Kincaid Grade Level score, which estimates a school grade.

Neither score tells us whether a review is positive, accurate, or well written!

In [ ]:
# textstat was installed when you ran `uv sync`.
from textstat import flesch_reading_ease

readability_score = flesch_reading_ease(sample_text)
print(f"Text: {sample_text} \nFlesch Reading Ease: {readability_score}")

## Stemming and Lemmatization

Both techniques try to reduce different forms of a word to a common form.

- Stemming cuts off endings using rules. The result might not be a real word.
- Lemmatization uses a dictionary and a part of speech. It can return a dictionary form, or leave the word unchanged if it doesn't find one.

One detail that matters: NLTK's lemmatizer assumes a **noun** unless we tell it otherwise. Let's compare that default with `pos='v'` (verb). `runner` is a noun, not another verb form of `run`.

[More on NLTK's lemmatizer](https://www.nltk.org/api/nltk.stem.wordnet.html).

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

sample_words = ['running', 'ran', 'runs', 'runner']
for word in sample_words:
    stemmed_word = stemmer.stem(word)
    noun_lemma = lemmatizer.lemmatize(word)
    verb_lemma = lemmatizer.lemmatize(word, pos='v')
    print(f'{word}: stem={stemmed_word}, noun lemma={noun_lemma}, verb lemma={verb_lemma}')

## Stop Words

Stop words are common words we might choose to remove. But be careful with sentiment: removing `not` from "not good" changes the meaning!

The code below shows what the standard list removes. Don't assume that removing all of these words will help your model. You could keep negations such as `not`, `no`, and `nor`, then compare the results.

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
filtered_tokens = [word for word in tokens if word.lower() not in stop_words]
# Show the first 30 tokens from each list
print(f"Original tokens (first 30): {tokens[:30]}")
print(f"Filtered tokens (first 30): {filtered_tokens[:30]}")

# 5. Challenge

Go back to the book reviews at the beginning of this notebook. The challenge is to build a better predictive model.

Some things you can try:

- use n-grams instead of just 1-grams
- try a stop-word list (remember the warning about negation!)
- use a stemmer or lemmatizer
- use a different machine learning model
- ...anything else you might imagine.

One detail: `CountVectorizer` and `TfidfVectorizer` already lowercase text and, with their default token pattern, ignore punctuation and single-character tokens. So "lowercase everything" alone won't change our current results.

Compare accuracy using the same evaluation approach. Keep track of what you tried; more preprocessing doesn't automatically mean a better model!